# <div style="color:white;display:fill;border-radius:5px;background-color:green;letter-spacing:0.1px;overflow:hidden"><p style="padding:20px;color:white;overflow:hidden;margin:0;font-size:100%;text-align:center">Import library</p>

In [1]:
import pandas as pd
import numpy as np
import xgboost
import pickle
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error
import os
from pathlib import Path

# <div style="color:white;display:fill;border-radius:5px;background-color:green;letter-spacing:0.1px;overflow:hidden"><p style="padding:20px;color:white;overflow:hidden;margin:0;font-size:100%;text-align:center">data_loading</p>

## 1.1. Load data

In [2]:
DATA_DIR = "./data/processed"

df_train = pd.read_csv(f"{DATA_DIR}/train.csv")
df_valid = pd.read_csv(f"{DATA_DIR}/valid.csv")
df_test = pd.read_csv(f"{DATA_DIR}/test.csv")

print('Train Shape: ', df_train.shape)
print('Valid Shape: ', df_valid.shape)
print('Test Shape: ', df_test.shape)

Train Shape:  (1168, 274)
Valid Shape:  (292, 274)
Test Shape:  (1459, 273)


## 1.2. Peak data

In [3]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1168 entries, 0 to 1167
Columns: 274 entries, MSSubClass to SalePrice
dtypes: float64(273), int64(1)
memory usage: 2.4 MB


In [4]:
df_valid.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 292 entries, 0 to 291
Columns: 274 entries, MSSubClass to SalePrice
dtypes: float64(273), int64(1)
memory usage: 625.2 KB


In [5]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1459 entries, 0 to 1458
Columns: 273 entries, MSSubClass to SaleCondition_Partial
dtypes: float64(273)
memory usage: 3.0 MB


# <div style="color:white;display:fill;border-radius:5px;background-color:green;letter-spacing:0.1px;overflow:hidden"><p style="padding:20px;color:white;overflow:hidden;margin:0;font-size:100%;text-align:center">data_preprocessing</p>

## 2.1. indicate feature

In [6]:
feature = [x for x in df_train.columns if x not in ['Id', 'SalePrice']]
target = 'SalePrice'

In [7]:
x_train = df_train[feature]
y_train = df_train[target]
x_valid = df_valid[feature]
y_valid = df_valid[target]
x_test  = df_test[feature]

# <div style="color:white;display:fill;border-radius:5px;background-color:green;letter-spacing:0.1px;overflow:hidden"><p style="padding:20px;color:white;overflow:hidden;margin:0;font-size:100%;text-align:center">data_transformation</p>

# <div style="color:white;display:fill;border-radius:5px;background-color:green;letter-spacing:0.1px;overflow:hidden"><p style="padding:20px;color:white;overflow:hidden;margin:0;font-size:100%;text-align:center">model_hypertuning</p>

In [8]:
xgb_model = xgboost.XGBRegressor(device='cuda',
                                 n_jobs=-1)
xgb_model.fit(x_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device='cuda', early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=None, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=None, n_jobs=-1,
             num_parallel_tree=None, random_state=None, ...)

In [9]:
param = {
    'n_estimators': [100, 500, 900, 1100, 1500],
    'max_depth': [2, 3, 5, 10, 15],
    'learning_rate': [0.05, 0.1, 0.15, 0.2],
    'min_child_weight': [1, 2, 3, 4],
    'booster': ['gbtree', 'gblinear'],
    'base_score': [0.25, 0.5, 0.75, 1],
}

In [10]:
random_cv = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param,
    cv=5, n_iter=50,
    scoring='neg_root_mean_squared_error', n_jobs=-1,
    verbose=5, return_train_score=True, random_state=42,
)
random_cv.fit(x_train, y_train)

Fitting 5 folds for each of 50 candidates, totalling 250 fits


RandomizedSearchCV(cv=5,
                   estimator=XGBRegressor(base_score=None, booster=None,
                                          callbacks=None,
                                          colsample_bylevel=None,
                                          colsample_bynode=None,
                                          colsample_bytree=None, device='cuda',
                                          early_stopping_rounds=None,
                                          enable_categorical=False,
                                          eval_metric=None, feature_types=None,
                                          gamma=None, grow_policy=None,
                                          importance_type=None,
                                          interaction_constraints=None,
                                          learning_rat...
                                          num_parallel_tree=None,
                                          random_state=None, ...),
                   n_iter=50, n_jobs=-1,
                   param_distributions={'base_score': [0.25, 0.5, 0.75, 1],
                                        'booster': ['gbtree', 'gblinear'],
                                        'learning_rate': [0.05, 0.1, 0.15, 0.2],
                                        'max_depth': [2, 3, 5, 10, 15],
                                        'min_child_weight': [1, 2, 3, 4],
                                        'n_estimators': [100, 500, 900, 1100,
                                                         1500]},
                   random_state=42, return_train_score=True,
                   scoring='neg_root_mean_squared_error', verbose=5)

In [11]:
print("Best parameters:")
print(random_cv.best_params_)


print("Best RMSE:")
print(-random_cv.best_score_)

Best parameters:
{'n_estimators': 900, 'min_child_weight': 1, 'max_depth': 2, 'learning_rate': 0.1, 'booster': 'gbtree', 'base_score': 0.25}
Best RMSE:
27045.365234375


In [12]:
random_cv.best_estimator_

XGBRegressor(base_score=0.25, booster='gbtree', callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device='cuda', early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.1, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=2, max_leaves=None,
             min_child_weight=1, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=900, n_jobs=-1,
             num_parallel_tree=None, random_state=None, ...)

# <div style="color:white;display:fill;border-radius:5px;background-color:green;letter-spacing:0.1px;overflow:hidden"><p style="padding:20px;color:white;overflow:hidden;margin:0;font-size:100%;text-align:center">model_training</p>

## 5.1. model_init

SVR

In [13]:
svr_model = SVR()

XgBoost

In [14]:
xgb_model = xgboost.XGBRegressor(
    base_score=0.25, booster='gbtree', colsample_bylevel=1,
    colsample_bynode=1, colsample_bytree=1, gamma=0,
    importance_type='gain', learning_rate=0.1,
    max_delta_step=0, max_depth=2, min_child_weight=1,
    n_estimators=900, n_jobs=-1, num_parallel_tree=1,
    objective='reg:squarederror', random_state=0, reg_alpha=0,
    reg_lambda=1, scale_pos_weight=1, subsample=1,
    tree_method='hist', validate_parameters=True,
    device='cuda',eval_metric='rmse'
)

RF

In [15]:
rf_model = RandomForestRegressor(random_state=42, n_jobs=-1)

## 5.2. model_fit

Decision Tree

In [16]:
svr_model.fit(x_train, y_train)

SVR()

XgBoost

In [17]:
xgb_model.fit(
    x_train,
    y_train,
    eval_set=[
        (x_valid, y_valid)
    ],
    verbose=True
)

[0]	validation_0-rmse:180944.26908
[1]	validation_0-rmse:164698.85895
[2]	validation_0-rmse:150290.33498
[3]	validation_0-rmse:137364.05521
[4]	validation_0-rmse:125833.41888
[5]	validation_0-rmse:115440.99926
[6]	validation_0-rmse:106309.92023
[7]	validation_0-rmse:98164.13023
[8]	validation_0-rmse:90862.60085
[9]	validation_0-rmse:84431.12308
[10]	validation_0-rmse:78576.07029
[11]	validation_0-rmse:73272.82315
[12]	validation_0-rmse:68920.77065
[13]	validation_0-rmse:64886.43904
[14]	validation_0-rmse:61424.08031
[15]	validation_0-rmse:58274.11488
[16]	validation_0-rmse:55389.89333
[17]	validation_0-rmse:52868.71428
[18]	validation_0-rmse:50621.02052
[19]	validation_0-rmse:48760.45628
[20]	validation_0-rmse:47167.70989
[21]	validation_0-rmse:45709.68691
[22]	validation_0-rmse:44406.06318
[23]	validation_0-rmse:43094.27598
[24]	validation_0-rmse:42096.06148
[25]	validation_0-rmse:41046.38713
[26]	validation_0-rmse:40110.31135
[27]	validation_0-rmse:39288.29202
[28]	validation_0-rmse:

XGBRegressor(base_score=0.25, booster='gbtree', callbacks=None,
             colsample_bylevel=1, colsample_bynode=1, colsample_bytree=1,
             device='cuda', early_stopping_rounds=None,
             enable_categorical=False, eval_metric='rmse', feature_types=None,
             gamma=0, grow_policy=None, importance_type='gain',
             interaction_constraints=None, learning_rate=0.1, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None, max_delta_step=0,
             max_depth=2, max_leaves=None, min_child_weight=1, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=900,
             n_jobs=-1, num_parallel_tree=1, random_state=0, ...)

RF

In [18]:
rf_model.fit(x_train, y_train)

RandomForestRegressor(n_jobs=-1, random_state=42)

## 5.3. model_predict

SVR

In [19]:
y_pred_svr = svr_model.predict(x_test)

XgBoost

In [20]:
y_pred_xgb = xgb_model.predict(x_test)

d:\Repositories\SGU_machine_learning_TEAM\.venv\lib\site-packages\xgboost\core.py:158: UserWarning: [22:04:11] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\common\error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)


ANN

In [21]:
y_pred_rf = rf_model.predict(x_test)

## 5.4. model_save

### 5.4.1. model

In [22]:
MODEL_DIR = Path("model")

os.makedirs(exist_ok = True,
            name     = MODEL_DIR)

SVR

In [23]:
f = MODEL_DIR / 'svr_model.pkl'
with open(f, 'wb') as model_file:
    pickle.dump(svr_model, model_file)
print(f'Đã lưu mô hình SVR: {f.resolve()}')

Đã lưu mô hình SVR: D:\Repositories\SGU_machine_learning_TEAM\week_4\practice\Lab04 - Feature Selection (Nhóm)\code\model\svr_model.pkl


XgBoost

In [24]:
f = MODEL_DIR / 'xgb_model.pkl'
with open(f, 'wb') as model_file:
    pickle.dump(xgb_model, model_file)
print(f'Đã lưu mô hình XGBoost: {f.resolve()}')

Đã lưu mô hình XGBoost: D:\Repositories\SGU_machine_learning_TEAM\week_4\practice\Lab04 - Feature Selection (Nhóm)\code\model\xgb_model.pkl


ANN

In [25]:
f = MODEL_DIR / 'rf_model.pkl'
with open(f, 'wb') as model_file:
    pickle.dump(rf_model, model_file)
print(f'Đã lưu mô hình Random Forest: {f.resolve()}')

Đã lưu mô hình Random Forest: D:\Repositories\SGU_machine_learning_TEAM\week_4\practice\Lab04 - Feature Selection (Nhóm)\code\model\rf_model.pkl


### 5.4.2. predict

In [26]:
OUTPUT_DIR = Path("model")

os.makedirs(exist_ok = True,
            name     = OUTPUT_DIR)

SVR

In [27]:
sub_df = pd.read_csv(f"data/sample_submission.csv")
sub_df['SalePrice'] = y_pred_svr
output_file = OUTPUT_DIR / 'sample_sub_svr.csv'
sub_df.to_csv(output_file, index=False)
print(f'Đã lưu submission SVR: {output_file.resolve()}')

Đã lưu submission SVR: D:\Repositories\SGU_machine_learning_TEAM\week_4\practice\Lab04 - Feature Selection (Nhóm)\code\model\sample_sub_svr.csv


XgBoost

In [28]:
sub_df = pd.read_csv(f"data/sample_submission.csv")
sub_df['SalePrice'] = y_pred_xgb
output_file = OUTPUT_DIR / 'sample_sub_xgb.csv'
sub_df.to_csv(output_file, index=False)
print(f'Đã lưu submission XGBoost: {output_file.resolve()}')

Đã lưu submission XGBoost: D:\Repositories\SGU_machine_learning_TEAM\week_4\practice\Lab04 - Feature Selection (Nhóm)\code\model\sample_sub_xgb.csv


RF

In [29]:
sub_df = pd.read_csv(f"data/sample_submission.csv")
sub_df['SalePrice'] = y_pred_rf
output_file = OUTPUT_DIR / 'sample_sub_rf.csv'
sub_df.to_csv(output_file, index=False)
print(f'Đã lưu submission Random Forest: {output_file.resolve()}')

Đã lưu submission Random Forest: D:\Repositories\SGU_machine_learning_TEAM\week_4\practice\Lab04 - Feature Selection (Nhóm)\code\model\sample_sub_rf.csv
